# Autoresearch SFT for the Nemotron Reasoning Challenge

> A reproducible, auditable BF16 LoRA SFT pipeline for `NVIDIA-Nemotron-3-Nano-30B-A3B-BF16`, built around an autonomous AI coding agent running Karpathy's **autoresearch** loop. Current best on `main`: **0.6000** vs the published Kaggle baseline of **0.5000**.

---

## What this notebook is

This is the public version of the work I've been doing on the [Nemotron Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge) — the full training pipeline, the contract that drives an autonomous agent's overnight runs, and a failure ledger of every non-trivial bug I have hit on this stack, so you do not have to rediscover them.

- **Repo:** [`github.com/priyanlc/autoresearch-sft-grpo`](https://github.com/priyanlc/autoresearch-sft-grpo)
- **Companion write-up:** [A framework for automating LLM fine-tuning with Autoresearch and OpenClaw](https://www.linkedin.com/feed/update/urn:li:ugcPost:7461543957576060928/)

---

## Why this might help your run

The pipeline is built for **single-A100 BF16 SFT** on a 30B-A3B MoE model with native `enable_thinking=True` reasoning traces, and respects the competition constraints:

- LoRA rank ≤ 32
- `\boxed{}` answer extraction
- Six puzzle categories
- Hidden-rules evaluation

The more interesting thing is what the repo carries *beside* the training script — an eight-artefact **chassis** that keeps the work honest over multi-day runs. Even if you do not run an autonomous agent, the chassis is useful infrastructure for a serious entry:

| Artefact | What you can use it for |
|---|---|
| `program.md` | A locked-decisions contract — what to optimise, what NOT to sweep, and why |
| `train.py` | A single, runnable BF16 SFT-only training script tuned for Nemotron-3-Nano |
| `prepare.py` + `data/val_split.json` | A fixed, deterministic, read-only 30-puzzle val split (5 per category, seed=0) |
| `FRICTION.md` | Twelve documented failure modes — the bugs I already debugged for you |
| `results.tsv` | One row per experiment, including failures, with per-category breakdowns |
| `STATUS.md` | A ~40-minute heartbeat ledger so the run is reconstructible from disk alone |
| `BRANCH_NOTES.md` | Per-branch config table and chronology |
| `git` history | Every change is reverted (not fixed-forward) on regressions |

---

## Three pain points already solved

If you are working with Nemotron-3-Nano BF16 on this stack, you will almost certainly hit at least one of these. The full debugging story for each lives in `FRICTION.md`:

**F-001 — Cache bugs that look like dead code.** Nemotron's caching layer is broken in four ways at once on this stack. The fix is two short lines in `train.py` that disable the fast path during eval. Evaluation is roughly **18× slower** with the cache off, but it is correct. 

**F-009 — The "optional" import that is not optional.** The model loader inspects the script statically before any runtime guards execute. Treat dependencies as install-time, not import-time. The repo has a pre-flight check that catches this before training starts.

**F-011 — Wrong answers that look "too round."** The chain-of-thought generator was producing nonsense for two puzzle categories. Aggregate metrics hid it. The contract requires per-category scores on every evaluation — that is how this got caught. Pull the same discipline into your eval and you will catch a class of bugs that aggregate scores hide.

---

## The metric trajectory on `main` (transparency)

Everything below is BF16 SFT-only on `main`. A sibling branch runs GRPO on top of the SFT-only base.

| Score | What it represents |
|---|---|
| 0.5000 | Published Kaggle baseline, no fine-tune |
| 0.5333 | First credible BF16 SFT-only build. The locked floor and revert target |
| 0.5667 | Baseline restoration after sorting infrastructure friction |
| 0.5333 | "Are they undertrained?" sweep. Reverted per branch hygiene |
| **0.6000** | F-011 regex fix. Current best — four samples on `gravity` above the variance floor |

Each jump came from finding a specific bug, not from running a hyperparameter sweep. The chassis is what made the bugs findable.

---

## The NVFP4 (Blackwell) sibling branch

A second sibling branch, **`nvfp4-blackwell`** (created 2026-04-12, forked from `main` at METRIC 0.5333), adapts the pipeline for Kaggle's **RTX PRO 6000 (Blackwell)** using NVIDIA's 4-bit floating-point format (NVFP4) via `FPQuantConfig`. The motivation is straightforward: BF16 base weights are ~60 GB; the NVFP4 base is ~17 GB. That headroom buys back larger batches, longer sequences, and the option to fit the 30B model on a single Blackwell card.

> **Status as of 2026-05-02: research-grade. No METRIC produced yet.** The branch's `results.tsv` is still empty. The pipeline reaches the SFT loop but trips on a documented integration seam between `transformers 5.7.0`, `peft 0.19.1`, `fp_quant 0.3.2`, and `qutlass 0.2.0`. The full debugging story for the current blocker is in this branch's own `FRICTION.md`.

### Configuration delta from `main`

| Parameter | `main` (BF16) | `nvfp4-blackwell` |
|---|---|---|
| `USE_NVFP4` | N/A | `True` |
| Base model size | ~60 GB | ~17 GB (NVFP4) |
| `BATCH_SIZE` | 1 | 2 |
| `SFT_MAX_SEQ_LEN` | 1024 | 2048 |
| `SFT_LR` | 2e-4 | 5e-5 |
| Optimiser | `adamw_torch` | `adamw_8bit` |
| `packing` | `False` | `True` |
| `MAX_GRAD_NORM` | 1.0 | 0.1 |
| `LORA_DROPOUT` | 0.05 | 0.0 (FPQuantLinear requires) |
| Synthetic data | None | 500/category (3,000 extra samples) |
| Reward shape | Binary + weighted | Cosine-scaled |

### Hardware and install requirements

- **Blackwell-class GPU** (RTX PRO 6000, B200, GB10). On Hopper/Ampere the NVFP4 path silently falls back to BF16 and the 30B model will not fit.
- **≥ 48 GB VRAM.** The dev box for this branch is the RTX PRO 6000 Blackwell Server Edition (95 GB). Peak VRAM observed at model load is **96.6 GB** on a 95 GB card — margins are slim.
- **CUDA 12.8+** for the FP4 kernels.
- **Real QuTLASS from GitHub** (IST-DASLab's CUTLASS-based NVFP4 kernels), not the PyPI stub of the same name (F-002 below).
- `mamba_ssm` and `causal_conv1d` CUDA extensions, built with `TORCH_CUDA_ARCH_LIST=12.0` to target `sm_120`. Full staged install in `runpod-setup.md`.

### Six integration failures already debugged on the branch

The branch's own `FRICTION.md` (numbering reset from `main`) has six entries so far. The ones most likely to bite a competitor on Blackwell hardware:

- **F-002 — `qutlass` on PyPI is an empty stub** that silently breaks the NVFP4 path. Install the real QuTLASS from GitHub with `--no-build-isolation`.
- **F-003 — `transformers.validate_quantization_for_training` rejects FPQuant + PEFT 0.19.1.** Patch: set `model._hf_peft_config_loaded = True` after `get_peft_model(...)`.
- **F-004 — RunPod `/workspace` is MooseFS-over-network** at ~33 MB/s. A 60 GB cold load takes ~22 minutes; the page cache mitigates after the first run (~3 min warm).
- **F-005 — `fp_quant` 0.3.2 has no backward for `FPQuant4x16NoMasterFn`** (the real-NVFP4 + no-master combo). Workaround: set `store_master_weights=True`, which keeps the real NVFP4 forward kernels but adds ~60 GB of BF16 master weights — total ~77 GB before LoRA + optimiser + activations.
- **F-006 — PEFT 0.19.1's `_replace_module` crashes on `qweight=None`** when master weights are enabled (current blocker). Candidate fix (untested): `del module.qweight` for any FPQuantLinear where `qweight is None`, before `get_peft_model`.

The pattern across all six is consistent: each of `transformers`, `peft`, `fp_quant`, and `qutlass` works in isolation, but the LoRA-on-FPQuant-master training path threads through all four and trips on seams no one has integration-tested. The `FPQuantLinear.__bases__` hack noted in `BRANCH_NOTES.md` looks like the first of several `train.py` workarounds the path needs.

### Why this might still help you

The branch has no metric yet, but the FRICTION ledger is the load-bearing artefact for any Blackwell competitor. F-002 through F-006 are deterministic on the pinned (`transformers 5.7.0` + `peft 0.19.1` + `fp_quant 0.3.2` + `qutlass 0.2.0`) version set, and each entry carries the upstream-file references that found the bug. If you are trying NVFP4 on this competition, those entries are likely to save you a working week.

```
git checkout nvfp4-blackwell
# then follow runpod-setup.md and BRANCH_NOTES.md
```

---

## How to use this in your own entry

Clone the repo and follow `README.md`. The full setup — `bootstrap.sh`, dependencies, the contract, the training script, and the current best LoRA adapter on `main` — is reproducible from disk.

```
git clone https://github.com/priyanlc/autoresearch-sft-grpo
cd autoresearch-sft-grpo
# then follow README.md
```

---

## Honest caveats

- The eval cache is **off** on purpose. Eval runs are slow. 
- The val split is 30 puzzles. Variance floor on the `gravity` category is roughly four samples. Small-set noise is real.
- LoRA rank is capped at 32 by the competition. The repo respects this.
- A run normally leaves behind only code changes and a metric. That is not enough — the failure ledger and per-category breakdowns are what make wins findable.

---

If you use any of this, I would love to hear what worked and what did not. The failure ledger is open to PRs.
